# Languages for AI

## What a processor understands

The **processor** (or **CPU**, *Central Processing Unit*) is the part of the computer that executes programs. It can only do very simple things, one small step at a time: read a number from memory, add or multiply two numbers, compare them, write a result back, jump to another step. What makes it powerful is speed: a modern CPU performs **billions** of these steps per second.

Those steps are stored as numbers, in a format called **machine code**. Each processor family has its own set of instructions: most PCs use *x86-64*, while phones and recent Apple computers use *ARM*. Nobody writes machine code by hand. We write in a **programming language**, and a tool translates it into machine code.

## High-level and low-level languages

Programming languages sit at different distances from the hardware:

- A **low-level language** such as **C** stays close to how the machine works. The programmer states the exact type of every variable (for example "a 64-bit integer") and manages memory directly.
- A **high-level language** such as **Python** hides those details. It offers lists, text, dictionaries and automatic memory management, so the programmer can think about the problem instead of the machine.

Either way, the program must end up as machine code before the CPU can run it. There are two classic ways of getting there: **compiling** and **interpreting**.

## Compiling and interpreting

Imagine a speech written in English that must reach an audience who only speak Spanish.

- **Translating the book**: a translator takes the whole text, translates it once, carefully, and prints the result. The translation takes time, but afterwards anyone can read it directly, as often as they like, with no translator present.
- **A live interpreter**: an interpreter stands next to the speaker and translates each sentence as it is spoken. The talk can start immediately, and the speaker can change the text on the fly. But every sentence is translated again every time the talk is given, and the interpreter must always be there.

Programming languages work the same way:

- A **compiler** translates the whole program into machine code *before* it runs. The result is an **executable**, a file that the CPU can run directly. C, C++, Rust and Go are usually compiled.
- An **interpreter** is a program that reads your program and carries it out step by step *while* it runs. Python, JavaScript and R are usually interpreted.

## A compiled program in C

To compare both strategies we use a small task: go through the numbers from 0 to ten million and add up the remainder of dividing each one by 7 (`i % 7`). The operation itself does not matter; it is just a cheap piece of arithmetic that must really be done for every number. (A simpler sum such as `total += i` would be a risky test: some C compilers, such as Clang, which is the compiler behind `gcc` on macOS, recognise that loop and replace it with the formula $n(n-1)/2$, leaving no loop to measure.)

This is the C version. `long long` declares a 64-bit integer (in C every variable needs a declared type), the `for` loop runs from `i = 0` while `i < count`, adding 1 each time (`i++`), and the program prints the total and the seconds the loop took.

In [ ]:
from pathlib import Path

print(Path("c/sum_loop.c").read_text())

// Adds up i % 7 for i = 0, 1, ..., count - 1 and prints the total and the seconds the loop took.
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

int main(int argc, char *argv[]) {
    long long count = atoll(argv[1]);

    struct timespec start, end;
    clock_gettime(CLOCK_MONOTONIC, &start);

    long long total = 0;
    for (long long i = 0; i < count; i++) {
        total += i % 7;
    }

    clock_gettime(CLOCK_MONOTONIC, &end);
    double seconds = (end.tv_sec - start.tv_sec) + (end.tv_nsec - start.tv_nsec) / 1e9;
    printf("%lld %f\n", total, seconds);
    return 0;
}



**Step 1: compile.** We run `gcc`, the GNU C compiler, from Python with `subprocess.run`, which runs a command as if it were typed in a terminal. `-O2` asks the compiler to *optimise*: since it translates in advance, it can take its time to look for faster machine code. `-o build/sum_loop` names the executable. A return code of `0` means the compilation succeeded.

In [ ]:
import subprocess

Path("build").mkdir(exist_ok=True)
compilation = subprocess.run(["gcc", "-O2", "c/sum_loop.c", "-o", "build/sum_loop"], capture_output=True, text=True)
compilation.returncode

0

The executable is not text. These are its first bytes, written as hexadecimal numbers: machine code (plus a header for the operating system) that only runs on this kind of processor and operating system.

In [ ]:
Path("build/sum_loop").read_bytes()[:32].hex(" ")

'7f 45 4c 46 02 01 01 00 00 00 00 00 00 00 00 00 03 00 3e 00 01 00 00 00 f0 11 00 00 00 00 00 00'

**Step 2: run.** The executable runs on its own, with no compiler involved:

In [ ]:
count = 10_000_000
c_run = subprocess.run(["build/sum_loop", str(count)], capture_output=True, text=True, check=True)
c_output = c_run.stdout.split()
c_total, c_seconds = int(c_output[0]), float(c_output[1])
c_total, c_seconds

(29999994, 0.005117)

## The same program in Python

In Python there is no separate compilation step: you write the code and run it straight away.

In [ ]:
def sum_of_remainders(count: int) -> int:
    total = 0
    for i in range(count):
        total += i % 7
    return total

In [ ]:
import time

start = time.perf_counter()
python_total = sum_of_remainders(count)
python_seconds = time.perf_counter() - start
python_total, python_seconds

(29999994, 0.1459981849984615)

In [ ]:
python_total == c_total, f"Python took {python_seconds / c_seconds:.0f} times longer than C"

(True, 'Python took 29 times longer than C')

Same result, same computer, same algorithm, and Python is *tens of times* slower. The reason is the live interpreter: for every step of the loop, ten million times, the Python interpreter has to work out again what the instruction means and check what kind of values it is working with (an integer? a decimal number? text?) before doing the actual arithmetic. The compiled C program did all that work once, before running, and the CPU now executes the loop directly.

→ Deep dive: [How programs run](deep-dives/how_programs_run.ipynb) shows the instructions the Python interpreter actually executes, how C catches type errors before running, and what the machine code looks like.

## Choosing between compiled and interpreted

| | Compiled (C, C++, Rust, Go) | Interpreted (Python, JavaScript, R) |
|---|---|---|
| Execution speed | Fast: the CPU runs machine code directly | Slower: every step passes through the interpreter |
| Development speed | Slower: more details to write, a compile step after every change | Faster: less code, run immediately |
| Error detection | Many errors caught before running | Found only when the faulty line runs |
| Portability | One executable per processor and operating system | The same script runs wherever the interpreter is installed |
| Interactivity | Write, compile, run | Run line by line: consoles, notebooks |

Neither is "better". The choice depends on **whose time is more expensive**. When code runs millions of times and every millisecond counts (operating systems, databases, games), the computer's time matters most, and compiled languages dominate. When the goal is to explore data and try ideas quickly, the programmer's time matters most, and interpreted languages dominate.

## Languages used in AI and machine learning

Some languages mix both strategies: they start by interpreting, and compile the most frequently used parts to machine code while the program runs. This is called **JIT** (*just-in-time*) compilation.

| Language | Compiled or interpreted | Role in AI and machine learning |
|---|---|---|
| Python | Interpreted | The main language: preparing data, training and using models, notebooks, research |
| C and C++ | Compiled | The engines inside AI libraries (NumPy, PyTorch, TensorFlow), and programs that run trained models on phones, cars and servers |
| CUDA | Compiled, for NVIDIA GPUs | Code that runs on graphics cards; the libraries behind GPU training (notebook 05) |
| R | Interpreted | Statistics and data analysis, especially in universities and life sciences |
| Julia | JIT-compiled | Scientific computing; a small AI community |
| Java, Scala | Compiled to an intermediate code, then JIT | Big-data platforms such as Spark, and enterprise systems that serve models |
| JavaScript | JIT-compiled | Running models inside web browsers |
| Rust | Compiled | Fast tools for the Python ecosystem (for example `uv`, which installed this project) |
| Mojo | Compiled | A new language (version 1.0 in 2026) with Python-like syntax, designed for fast AI code |

→ Deep dive: [Languages for AI beyond Python](deep-dives/language_landscape.md) looks at each of these in more detail.

## Why Python is the main language of AI

- **It is easy to read and write.** Python code looks close to plain pseudocode, so scientists, analysts and students who are not professional programmers can use it.
- **It is interactive.** Notebooks like this one let you run a piece of code and see the result immediately, which is exactly what exploring data and testing ideas requires.
- **It has the ecosystem.** NumPy, pandas, scikit-learn, PyTorch, TensorFlow and Hugging Face are all used from Python. New models, tutorials and research code are published in Python first.
- **It has the community.** Because everyone uses it, most courses, answers and examples are in Python, which attracts even more users.

But the experiment above leaves a puzzle. Python is tens of times slower than C on a simple loop, and training an AI model takes trillions of arithmetic operations. How can one of the slowest languages be the main language of AI? The rest of this series answers that question.

## Key ideas

- A CPU only runs **machine code**; programming languages must be translated into it.
- A **compiler** translates the whole program in advance; an **interpreter** carries it out step by step while it runs. Compiled code runs much faster; interpreted code is quicker to write and try.
- AI uses several languages, but **Python** is the main one because it is easy, interactive and has the biggest ecosystem.
- Python's slowness is a real puzzle for a field that needs trillions of operations.